In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings('ignore')

image_emb = np.load('image_emb.npy', allow_pickle=True)
image_labels = np.load('image_labels.npy', allow_pickle=True)
image_emb_test = np.load('image_emb_test.npy', allow_pickle=True)
image_labels_test = np.load('image_labels_test.npy', allow_pickle=True)

k = 10 
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

cluster_assignments = kmeans.fit_predict(image_emb)

cluster_to_label = {}

for cluster_id in range(k):
    # znajdź indeksy obrazów które wpadły do obecnego klastra
    idx_in_cluster = np.where(cluster_assignments == cluster_id)[0]
    
    # wyciągnij etykiety dla tych konkretnych obrazów
    labels_in_cluster = image_labels[idx_in_cluster]
    
    # krok kluczowy: odrzucamy puste etykiety (nieoznaczone zdjęcia to zazwyczaj puste stringi '')
    labeled_only = labels_in_cluster[labels_in_cluster != '']
    
    if len(labeled_only) > 0:
        #szukamy najczęstszej etykiety w klastrze
        unique_vals, counts = np.unique(labeled_only, return_counts=True)
        most_frequent_label = unique_vals[np.argmax(counts)]
        cluster_to_label[cluster_id] = most_frequent_label
    else:
        cluster_to_label[cluster_id] = "unknown"

print("Zmapowane klastry na klasy:", cluster_to_label)

test_cluster_assignments = kmeans.predict(image_emb_test)

y_pred_test = np.array([cluster_to_label.get(c, "unknown") for c in test_cluster_assignments])

acc_test = accuracy_score(image_labels_test, y_pred_test)
print(f"Dokładność (Accuracy) na zbiorze testowym: {acc_test:.4f}")